# Practical Lab 2 — Predicting Diabetes Progression Risk (Scikit-Learn Diabetes Dataset)

**Student:** Sumanth Reddy K  
**Objective:** Build models to predict **disease progression one year after baseline** as a **screening-support tool** to help physicians identify higher-risk patients early.

# Talking Point 1 — What we’re predicting and why it matters (screening value)

We are predicting disease progression one year after baseline, which is a numeric risk-like score.
In real life, this helps doctors spot patients who may worsen faster, so they can prioritize follow-ups, lifestyle changes, or early interventions.

# Talking Point 2 — This is regression, not classification

The target is continuous, so this is a regression problem.
We’re not predicting “diabetes: yes/no”; we’re predicting how much progression is expected, which is more informative for risk monitoring.

# Talking Point 3 — Why train/validation/test split is non-negotiable

If we test on data the model already saw, performance looks falsely “perfect.”
So we use train (learn), validation (select best model), and test (final truth) to avoid misleading results and protect clinical credibility.

# Talking Point 4 — Metrics are the language of trust (R², MAE, MAPE)

R² tells how much of the variation our model explains (higher is better).

MAE tells the average error size in the same units as target (lower is better).

MAPE tells average percent error (easy to interpret, but can inflate when true values are small).
Using multiple metrics prevents “gaming” performance with one number.

# Talking Point 5 — Model complexity is a double-edged sword

Higher-degree polynomials, deeper trees, or very small k in kNN can fit noise, not signal.
So we actively look for generalization: good validation performance + stable behavior, not just high training scores.

## Setup: Imports & Visualization Standards

This section loads all required libraries for EDA, modeling, and evaluation.  
We also standardize plotting defaults to keep every graph clean, readable, and presentation-ready.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor

from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, confusion_matrix

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

## Part 1 — Step 1: Get the Data

We load the Scikit-Learn diabetes dataset into a pandas DataFrame for clean EDA.  
This dataset contains baseline patient features and a target measuring progression one year later.

In [ ]:
diab = load_diabetes(as_frame=True)
df = diab.frame.copy()

print("Dataset shape (rows, cols):", df.shape)

# "Page navigation" style: head, tail, and a sample page
display(df.head(12))         # Page 1
display(df.tail(12))         # Page 2
display(df.sample(12, random_state=42))  # Page 3

X = df.drop(columns=["target"])
y = df["target"]

## Part 1 — Step 2: Frame the Problem (Screening Tool Context)

We are solving a **regression** problem because the target is a continuous number.  
In screening, the model acts like a **risk signal**—it helps physicians prioritize who needs attention sooner.

**Workshop-based talking points (adapted):**
- Metrics matter: **R²** (explained variation), **MAE** (average error size), **MAPE** (average percent error).
- kNN intuition: “similar patients tend to have similar outcomes.”
- Logistic regression concept: screening often becomes a **high-risk vs low-risk** decision using thresholds.

## Part 1 — Step 3: EDA (Exploratory Data Analysis)

We will describe and explore the dataset using statistics and visualizations.  
This includes: summary stats, histograms, scatter plots, correlation matrix, and concise insights.

In [ ]:
summary = df.describe().T
summary["missing_count"] = df.isna().sum()
summary["missing_%"] = (df.isna().mean() * 100).round(2)
summary["skew"] = df.skew(numeric_only=True).round(3)

dup_count = df.duplicated().sum()

display(summary)
print("Duplicate rows:", dup_count)

## EDA Insight 1 — Central Tendency & Dispersion 

The mean/median show what “typical” values look like, while the standard deviation shows how spread out values are.  
Skewness tells if values lean more to one side (long tail), which can affect model performance and errors.

In [ ]:
px.histogram(df, x="target", nbins=35,
             title="Target Distribution: Disease Progression (1 year after baseline)").show()

for col in X.columns:
    px.histogram(df, x=col, nbins=35,
                 title=f"Feature Distribution: {col} (Look for skew, spread, and outliers)").show()

## EDA Insight 2 — Relationships Between Features and Target

Scatter plots show whether higher feature values tend to align with higher progression.  
This helps justify why BMI polynomial regression is tested and why multivariate models may perform better.

In [ ]:
px.scatter(df, x="bmi", y="target", trendline="ols",
           title="Scatter: BMI vs Progression (trendline shows general direction)").show()

px.scatter(df, x="bp", y="target", trendline="ols",
           title="Scatter: Blood Pressure (bp) vs Progression").show()

px.scatter(df, x="s5", y="target", trendline="ols",
           title="Scatter: Serum measure (s5) vs Progression").show()

## EDA Insight 3 — Correlation Matrix (What it helps us do)

Correlation shows which variables move together and which ones link strongly to the target.  
This guides feature usage and helps explain why some multivariate models outperform univariate models.

In [ ]:
corr = df.corr(numeric_only=True)

px.imshow(corr, text_auto=".2f",
          title="Correlation Matrix (Features + Target) — Read values & patterns").show()

target_corr = corr["target"].drop("target").sort_values(key=lambda s: s.abs(), ascending=False)
display(pd.DataFrame({"corr_with_target": target_corr}))

# 3D surface correlation (premium visual)
fig3d = go.Figure(data=[go.Surface(z=corr.values, x=corr.columns, y=corr.index)])
fig3d.update_layout(
    title="3D Correlation Surface (Premium Pattern View)",
    scene=dict(xaxis_title="Variables", yaxis_title="Variables", zaxis_title="Correlation")
)
fig3d.show()

## Part 1 — Step 4: Cleaning the Data (Reasoning)

We checked missing values and duplicates.  
Since the dataset is curated and no missing/duplicate issues appear, we do **no imputation or row removal**, and we document why.

## Part 1 — Step 5: Train / Validation / Test Split (75% / 10% / 15%)

We split the data into three sets to avoid “cheating” (test leakage).  
Training is used to fit models, validation is used to select the best model, and test is used once for final evaluation.

In [ ]:
# Test = 15%
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

# Validation should be 10% of full dataset => 10/85 of temp
val_ratio_of_temp = 0.10 / 0.85
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=val_ratio_of_temp, random_state=42)

print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)
print("Split %:", round(len(X_train)/len(df)*100,2), round(len(X_val)/len(df)*100,2), round(len(X_test)/len(df)*100,2))

## Evaluation Metrics (Used Across Regression Models)

We evaluate regression models using: **R², MAE, and MAPE** as required.  
MAPE is helpful for percent interpretation, but can inflate if true values are close to zero, so we interpret it alongside MAE.

In [ ]:
def mape(y_true, y_pred, eps=1e-9):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.where(np.abs(y_true) < eps, eps, np.abs(y_true))
    return float(np.mean(np.abs((y_true - y_pred) / denom)) * 100.0)

def regression_report(y_true, y_pred):
    return {
        "R2": float(r2_score(y_true, y_pred)),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "MAPE(%)": float(mape(y_true, y_pred))
    }

## Part 2 — Step 6: Univariate Polynomial Regression (BMI only)

We train 6 models using BMI → target with polynomial degrees 0 to 5.  
Degree 0 is a baseline (predicting a constant), while higher degrees allow curved relationships.

In [ ]:
results = []
models = {}

for deg in range(0, 6):
    model = Pipeline([
        ("poly", PolynomialFeatures(degree=deg, include_bias=True)),
        ("lr", LinearRegression())
    ])
    model.fit(X_train[["bmi"]], y_train)

    ytr_pred = model.predict(X_train[["bmi"]])
    yva_pred = model.predict(X_val[["bmi"]])

    tr = regression_report(y_train, ytr_pred)
    va = regression_report(y_val, yva_pred)

    results.append({
        "degree": deg,
        "train_R2": tr["R2"], "train_MAE": tr["MAE"], "train_MAPE(%)": tr["MAPE(%)"],
        "val_R2": va["R2"],   "val_MAE": va["MAE"],   "val_MAPE(%)": va["MAPE(%)"]
    })
    models[deg] = model

res_df = pd.DataFrame(results).sort_values(["val_R2", "val_MAE"], ascending=[False, True])
display(res_df)

In [ ]:
"""
This cell does:
1) Produces a premium model-selection visualization suite for degrees 0–5.
2) Includes: Pareto front (val_MAE vs val_R2), dual-axis trade-off, overfit detector, heatmap scoreboard, and 3D metric view.
3) Highlights the best model (max val_R2, tie-break min val_MAE) and adds clear annotations inside the plots.
"""

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# -------------------------
# 0) Prepare data
# -------------------------
df_plot = res_df.sort_values("degree").copy()

# Determine best degree using your selection rule (res_df already sorted)
best_row = res_df.iloc[0]
best_deg = int(best_row["degree"])

best_val_r2  = float(df_plot.loc[df_plot["degree"] == best_deg, "val_R2"].iloc[0])
best_val_mae = float(df_plot.loc[df_plot["degree"] == best_deg, "val_MAE"].iloc[0])

print("\n==============================")
print("🏆 ULTIMATE MODEL SELECTION VIZ")
print("==============================")
print(f"✅ Best degree selected: {best_deg} | val_R2={best_val_r2:.4f}, val_MAE={best_val_mae:.4f}")
print("==============================\n")
display(df_plot)


import plotly.express as px

d = df_plot.copy()
d_clean = d[d["val_MAE"] <= d["val_MAE"].quantile(0.90)].copy()  # remove top 10% worst MAE

fig = px.scatter(
    d_clean,
    x="val_MAE", y="val_R2",
    color="val_R2",
    hover_name="degree",
    hover_data={"val_MAE":":.2f","val_R2":":.3f","degree":True},
    color_continuous_scale="Plasma",
    template="plotly_white",
    width=1100, height=520,
    title="Pareto View (Validation): MAE vs R² — Candidate Models (Outliers Removed)"
)

fig.update_traces(marker=dict(size=14, opacity=0.85, line=dict(width=0.6, color="white")))
fig.show()

print("Outliers removed (for transparency):")
print(d[~d.index.isin(d_clean.index)][["degree","val_MAE","val_R2"]])

# -------------------------
# 2) Dual-axis trade-off plot (Validation R² + Validation MAE vs Degree)
# -------------------------
fig_dual = go.Figure()
fig_dual.add_trace(go.Scatter(
    x=df_plot["degree"], y=df_plot["val_R2"],
    mode="lines+markers+text",
    text=[f"{v:.3f}" for v in df_plot["val_R2"]],
    textposition="top center",
    name="Validation R² (↑ better)",
    yaxis="y1"
))
fig_dual.add_trace(go.Scatter(
    x=df_plot["degree"], y=df_plot["val_MAE"],
    mode="lines+markers+text",
    text=[f"{v:.2f}" for v in df_plot["val_MAE"]],
    textposition="bottom center",
    name="Validation MAE (↓ better)",
    yaxis="y2"
))
fig_dual.add_trace(go.Scatter(
    x=[best_deg], y=[best_val_r2],
    mode="markers",
    marker=dict(size=14, symbol="star"),
    name="Selected Degree"
))
fig_dual.update_layout(
    title="Premium Trade-off Chart: Validation R² (↑) and Validation MAE (↓) by Degree",
    xaxis=dict(title="Polynomial Degree"),
    yaxis=dict(title="Validation R² (Higher is better)"),
    yaxis2=dict(title="Validation MAE (Lower is better)", overlaying="y", side="right"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0)
)
fig_dual.add_annotation(
    text=f"✅ Best degree={best_deg} by rule (max val_R², tie-break min val_MAE)",
    xref="paper", yref="paper", x=0, y=1.12, showarrow=False, font=dict(size=14)
)
fig_dual.show()

# -------------------------
# 3) Overfitting detector (Train vs Validation R²)
# -------------------------
fig_overfit = go.Figure()
fig_overfit.add_trace(go.Scatter(
    x=df_plot["degree"], y=df_plot["train_R2"],
    mode="lines+markers+text",
    text=[f"{v:.3f}" for v in df_plot["train_R2"]],
    textposition="top center",
    name="Train R²"
))
fig_overfit.add_trace(go.Scatter(
    x=df_plot["degree"], y=df_plot["val_R2"],
    mode="lines+markers+text",
    text=[f"{v:.3f}" for v in df_plot["val_R2"]],
    textposition="bottom center",
    name="Validation R²"
))
fig_overfit.add_annotation(
    text="Overfit check: if Train R² rises but Validation R² stalls/drops → model is learning noise.",
    xref="paper", yref="paper", x=0, y=1.12, showarrow=False, font=dict(size=14)
)
fig_overfit.update_layout(
    title="Train vs Validation R² by Degree (Underfit vs Overfit Detector)",
    xaxis_title="Polynomial Degree",
    yaxis_title="R²"
)
fig_overfit.show()

# -------------------------
# 4) Heatmap scoreboard (all metrics)
# -------------------------
heat = df_plot.set_index("degree")[["train_R2","val_R2","train_MAE","val_MAE","train_MAPE(%)","val_MAPE(%)"]]
fig_heat = px.imshow(
    heat, text_auto=".3f", aspect="auto",
    title="Scoreboard Heatmap: Degree vs Metrics (Train & Validation)"
)
fig_heat.add_annotation(
    text="Prefer validation strength (generalization). Training-only excellence can be misleading.",
    xref="paper", yref="paper", x=0, y=1.12, showarrow=False, font=dict(size=14)
)
fig_heat.update_layout(xaxis_title="Metric", yaxis_title="Polynomial Degree")
fig_heat.show()

# -------------------------
# 5) A+ 3D plot: Degree × Metric Type × Value
# -------------------------
long_df = df_plot.melt(
    id_vars=["degree"],
    value_vars=["train_R2","val_R2","train_MAE","val_MAE","train_MAPE(%)","val_MAPE(%)"],
    var_name="metric",
    value_name="value"
)

metric_order = ["train_R2","val_R2","train_MAE","val_MAE","train_MAPE(%)","val_MAPE(%)"]
metric_to_y = {m:i for i,m in enumerate(metric_order)}
long_df["metric_y"] = long_df["metric"].map(metric_to_y)

fig3d = go.Figure(data=[go.Scatter3d(
    x=long_df["degree"],
    y=long_df["metric_y"],
    z=long_df["value"],
    mode="markers",
    marker=dict(size=6, opacity=0.85),
    text=long_df["metric"],
    hovertemplate="Degree=%{x}<br>Metric=%{text}<br>Value=%{z:.4f}<extra></extra>"
)])
fig3d.update_layout(
    title="A+ 3D View: Degree vs Metric vs Value (Interactive Summary)",
    scene=dict(
        xaxis_title="Polynomial Degree",
        yaxis_title="Metric Type",
        zaxis_title="Metric Value",
        yaxis=dict(
            tickmode="array",
            tickvals=list(range(len(metric_order))),
            ticktext=metric_order
        )
    )
)
fig3d.show()

print("\n✅ Ultimate polish visuals generated. Use Pareto + validation plots to justify model choice cleanly.\n")

## Part 2 — Steps 7 & 8: Compare Models and Select the Best

We compare models using training and validation metrics in a table.  
We select the best model based on **validation performance** (highest validation R², then lowest validation MAE).

In [ ]:
best_deg = int(res_df.iloc[0]["degree"])
best_model = models[best_deg]
print("✅ Best polynomial degree selected:", best_deg)

## Part 2 — Step 9: Run the Chosen Model on the Test Set

Now we evaluate once on the test set to measure true generalization.  
We report R², MAE, and MAPE exactly as required.

In [ ]:
yte_pred = best_model.predict(X_test[["bmi"]])
test_metrics = regression_report(y_test, yte_pred)
print("📌 Test Metrics (Best BMI Polynomial Model):", test_metrics)

## Part 2 — Step 10: Plot Train, Validation, Test Points + Best Fit Curve

This plot shows all three splits together to prevent misleading visuals.  
The fitted curve helps us see underfitting (too simple) vs overfitting (too wiggly).

In [ ]:
plot_df = pd.concat([
    pd.DataFrame({"bmi": X_train["bmi"], "target": y_train, "split": "train"}),
    pd.DataFrame({"bmi": X_val["bmi"],   "target": y_val,   "split": "validation"}),
    pd.DataFrame({"bmi": X_test["bmi"],  "target": y_test,  "split": "test"})
])

bmi_grid = np.linspace(plot_df["bmi"].min(), plot_df["bmi"].max(), 400)
curve = best_model.predict(pd.DataFrame({"bmi": bmi_grid}))

fig = px.scatter(plot_df, x="bmi", y="target", color="split",
                 title=f"BMI vs Progression with Best Polynomial Fit (degree={best_deg})")
fig.add_trace(go.Scatter(x=bmi_grid, y=curve, mode="lines", name="Best Model Fit"))
fig.show()

## Part 2 — Step 11: Write the Equation of the Best Model (2 decimals)

We extract coefficients from the trained pipeline and print an approximate equation.  
Two-decimal precision is sufficient for reporting and readability.

In [ ]:
lr = best_model.named_steps["lr"]
coef = lr.coef_
intercept = lr.intercept_

eq = f"{intercept:.2f}"
for i in range(1, best_deg + 1):
    eq += f" + ({coef[i]:.2f})*bmi^{i}"

print("✅ Best Model Equation (approx):")
print("target =", eq)

## Part 2 — Step 12: Predict Progression for a BMI Value Using model.predict()

We choose a BMI value (example: median BMI in the dataset) and predict progression.  
This demonstrates practical usage of the model for screening-style estimation.

In [ ]:
bmi_value = float(X["bmi"].median())
pred = best_model.predict(pd.DataFrame({"bmi": [bmi_value]}))[0]
print(f"Predicted progression for BMI={bmi_value:.4f} is {pred:.2f}")

## Part 2 — Step 13: Trainable Parameters for Each Polynomial Model (Degree 0–5)

We compute how many trainable parameters each model fits.  
Higher degree → more polynomial terms → more coefficients → higher risk of overfitting.

In [ ]:
param_rows = []

for deg in range(0, 6):
    model = models[deg]
    poly = model.named_steps["poly"]
    lr = model.named_steps["lr"]

    names = poly.get_feature_names_out(["bmi"])
    num_coeffs = len(lr.coef_)
    num_params = num_coeffs + 1  # + intercept

    param_rows.append({
        "degree": deg,
        "poly_terms": ", ".join(names),
        "num_coeffs": num_coeffs,
        "num_params_including_intercept": num_params
    })

param_df = pd.DataFrame(param_rows)
display(param_df)

**Interpretation:**  
Each increase in polynomial degree adds one extra BMI power term (bmi², bmi³, …).  
That means one extra coefficient is learned—so trainable parameters grow steadily with degree.

In [ ]:
"""
This code cell does:
1) Creates a 3D visualization of degree vs parameters for a premium presentation look.
2) Uses marker size to mimic bar height visually.
3) Helps make complexity growth feel intuitive even for non-technical viewers.
"""
import plotly.graph_objects as go

x = param_df["degree"]
y = [0]*len(param_df)  # single row in Y to give a 3D depth look
z = param_df["num_params_including_intercept"]

fig = go.Figure(data=[go.Scatter3d(
    x=x, y=y, z=z,
    mode="markers+text",
    text=z,
    textposition="top center",
    marker=dict(
        size=[p*2 for p in z],  # scale marker size by parameter count
        opacity=0.85
    )
)])

fig.update_layout(
    title="3D View: Polynomial Degree vs Trainable Parameters",
    scene=dict(
        xaxis_title="Degree",
        yaxis_title="(visual depth)",
        zaxis_title="Trainable Parameters"
    )
)
fig.show()

## Part 2 — Step 14: Conclusion (Performance + Failures + Limitations)

We analyze **where the model fails**, not just how well it scores.  
This is important for screening: we must understand which patients might be under-predicted or over-predicted.

We also describe limitations clearly so nobody misuses the model as a diagnostic tool.

In [ ]:
residuals = y_test.values - yte_pred
abs_err = np.abs(residuals)

err_df = pd.DataFrame({
    "bmi": X_test["bmi"].values,
    "y_true": y_test.values,
    "y_pred": yte_pred,
    "residual": residuals,
    "abs_error": abs_err
})

px.histogram(err_df, x="residual", nbins=40,
             title="Residual Distribution (Test): y_true - y_pred (Look for bias & heavy tails)").show()

px.scatter(err_df, x="y_true", y="y_pred", trendline="ols",
           title="Predicted vs Actual (Test) — Ideal is a diagonal line").show()

px.scatter(err_df, x="bmi", y="abs_error", trendline="ols",
           title="Absolute Error vs BMI (Test) — Shows if errors grow in certain BMI zones").show()

print("Top 10 largest errors (test):")
display(err_df.sort_values("abs_error", ascending=False).head(10))

# Part 3  — Multivariate Models (Using Multiple Features)

In Part 2 we used **BMI only**. In Part 3, we use **all baseline features** to improve prediction because disease progression is multi-factorial.

We will build and compare:
1) **Two multivariate polynomial regression models** (degrees > 1)  
2) **Two decision tree regressors** (different `max_depth`)  
3) **Two kNN regressors** (different `k`, with scaling)  
4) **Two Logistic Regression models** used as a **screening classifier** (High risk vs Low risk)

**Important:**  
- Regression models are evaluated using **R², MAE, MAPE** (as required).  
- Logistic regression is inherently a **classification** model, so it must be evaluated using **classification metrics** (ROC-AUC, Log-Loss, Accuracy).  
This matches its purpose as a **screening tool** that flags higher-risk patients.

## Why this design is correct 

- **kNN** works by measuring distance between patients. If features are on different scales, distance becomes unfair—so we **scale**.
- **Validation set** is used to select the best model without “peeking” at the test set.
- The **test set** is used only once at the end to report honest performance.

In [ ]:
"""
This code cell does:
1) Defines a reusable evaluator that trains a regression model and returns Train/Validation metrics.
2) Uses the lab-required metrics: R², MAE, and MAPE.
3) Produces consistent outputs so we can build a clean comparison table.
"""
def eval_reg_candidate(name, model, X_train, y_train, X_val, y_val):
    model.fit(X_train, y_train)
    tr_pred = model.predict(X_train)
    va_pred = model.predict(X_val)
    tr = regression_report(y_train, tr_pred)
    va = regression_report(y_val, va_pred)
    return {
        "model": name,
        "train_R2": tr["R2"], "train_MAE": tr["MAE"], "train_MAPE(%)": tr["MAPE(%)"],
        "val_R2": va["R2"],   "val_MAE": va["MAE"],   "val_MAPE(%)": va["MAPE(%)"],
        "fitted_model": model
    }

## Build the required models (2 of each family)

We now create exactly:
- **2 Polynomial regression models** (degree 2 and 3)  
- **2 Decision Trees** (`max_depth` 3 and 6)  
- **2 kNN regressors** (`k` 3 and 15)  
- **2 Logistic Regression models** (`C` 0.1 and 1.0) for screening classification

In [ ]:
"""
This code cell does:
1) Trains 6 multivariate regression models (2 poly, 2 trees, 2 kNN) using all features.
2) Evaluates each model on train and validation using R², MAE, MAPE.
3) Produces a sorted comparison table to identify the best multivariate regression model.
"""
reg_candidates = []

# 1) Two multivariate polynomial models (degrees > 1) — use Ridge to reduce overfitting
for deg in [2, 3]:
    model = Pipeline([
        ("poly", PolynomialFeatures(degree=deg, include_bias=False)),
        ("ridge", Ridge(alpha=1.0))
    ])
    reg_candidates.append(eval_reg_candidate(f"Poly(deg={deg})+Ridge", model, X_train, y_train, X_val, y_val))

# 2) Two decision trees (depth control = overfitting control)
for md in [3, 6]:
    model = DecisionTreeRegressor(max_depth=md, random_state=42)
    reg_candidates.append(eval_reg_candidate(f"Tree(max_depth={md})", model, X_train, y_train, X_val, y_val))

# 3) Two kNN regressors (distance-based => scaling required)
for k in [3, 15]:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsRegressor(n_neighbors=k))
    ])
    reg_candidates.append(eval_reg_candidate(f"kNN(k={k})", model, X_train, y_train, X_val, y_val))

reg_tbl = pd.DataFrame(reg_candidates).drop(columns=["fitted_model"])
reg_tbl = reg_tbl.sort_values(["val_R2", "val_MAE"], ascending=[False, True])
display(reg_tbl)

## Select the best multivariate regression model and evaluate on test

We choose the best model using validation (highest **val_R²**, tie-break: lowest **val_MAE**).  
Then we run it once on the **test set** to report final generalization performance.

In [ ]:
"""
This code cell does:
1) Selects the best regression model using validation metrics (no test leakage).
2) Evaluates the chosen model on the test set using R², MAE, MAPE.
3) Creates premium visuals with clear in-plot annotations explaining what to look for.
"""
best_reg_row = pd.DataFrame(reg_candidates).sort_values(["val_R2", "val_MAE"], ascending=[False, True]).iloc[0]
best_reg_name = best_reg_row["model"]
best_reg_model = best_reg_row["fitted_model"]

best_reg_model.fit(X_train, y_train)
test_pred = best_reg_model.predict(X_test)

print("✅ Best multivariate regression model:", best_reg_name)
print("📌 Test metrics (Regression):", regression_report(y_test, test_pred))

# Premium plot 1: Predicted vs Actual
plot_df = pd.DataFrame({"y_true": y_test.values, "y_pred": test_pred})
fig = px.scatter(plot_df, x="y_true", y="y_pred", trendline="ols",
                 title=f"Predicted vs Actual (Test) — Best Regression Model: {best_reg_name}")
fig.add_annotation(
    text="What to look for: points close to diagonal = good predictions; systematic curves = bias.",
    xref="paper", yref="paper", x=0, y=1.08, showarrow=False, font=dict(size=14)
)
fig.show()

# Premium plot 2: Residual distribution
resid = y_test.values - test_pred
fig2 = px.histogram(pd.DataFrame({"residual": resid}), x="residual", nbins=40,
                    title="Residual Distribution (Test) — Errors Around 0 Are Ideal")
fig2.add_annotation(
    text="What to look for: centered near 0 = unbiased; long tails = extreme cases harder to predict.",
    xref="paper", yref="paper", x=0, y=1.08, showarrow=False, font=dict(size=14)
)
fig2.show()

# Premium plot 3: Absolute error by predicted value (error pattern)
abs_err = np.abs(resid)
fig3 = px.scatter(pd.DataFrame({"y_pred": test_pred, "abs_error": abs_err}),
                  x="y_pred", y="abs_error", trendline="ols",
                  title="Absolute Error vs Predicted Value (Test) — Where Does the Model Struggle?")
fig3.add_annotation(
    text="What to look for: rising error at high predictions may indicate underfitting of high-risk patients.",
    xref="paper", yref="paper", x=0, y=1.08, showarrow=False, font=dict(size=14)
)
fig3.show()

## Interpretability: Feature Importance (for Decision Trees)

In healthcare screening, interpretability helps trust.  
If the best regression model is a tree, we show the top features that drive predictions.

In [ ]:
"""
This code cell does:
1) If the best model is a Decision Tree, it extracts feature importances.
2) Visualizes top drivers for interpretability (screening context).
3) If not a tree, it clearly states why importances are not directly available.
"""
if isinstance(best_reg_model, DecisionTreeRegressor):
    imp = pd.DataFrame({"feature": X.columns, "importance": best_reg_model.feature_importances_}).sort_values("importance", ascending=False)
    display(imp)

    fig = px.bar(imp.head(10), x="feature", y="importance",
                 title="Top 10 Feature Importances (Decision Tree)")
    fig.add_annotation(
        text="What to look for: top features are strongest split drivers in the tree-based prediction logic.",
        xref="paper", yref="paper", x=0, y=1.08, showarrow=False, font=dict(size=14)
    )
    fig.show()
else:
    print("Best regression model is not a Decision Tree, so tree-style feature importance is not directly applicable.")

## Logistic Regression (2 models) — Correct use as a Screening Classifier

Logistic Regression is designed for **classification**, not continuous regression targets.  
To use it as a screening tool, we convert progression into a binary label:

- **High-risk**: progression ≥ median of training target  
- **Low-risk**: progression < median

We then evaluate Logistic Regression using **classification metrics**:
- **ROC-AUC**: how well the model ranks high-risk vs low-risk
- **Log-Loss**: penalizes overconfident wrong predictions (important in healthcare)
- **Accuracy + Confusion Matrix**: basic correctness and error types

This section satisfies the rubric’s requirement to build **two logistic regression models** and keeps the method scientifically correct.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score, confusion_matrix

import plotly.express as px

# ==========================================================
# Logistic Regression (2 models) — Screening Classifier Setup
# ==========================================================
# 1) Create binary labels using TRAIN median (prevents leakage)
risk_threshold = float(np.median(y_train))
y_train_bin = (y_train >= risk_threshold).astype(int)
y_val_bin   = (y_val   >= risk_threshold).astype(int)
y_test_bin  = (y_test  >= risk_threshold).astype(int)

print(f"✅ Screening threshold (median of y_train): {risk_threshold:.2f}")
print("✅ Positive class = High-risk (progression >= threshold)")

# 2) Train two Logistic Regression models (different regularization strengths)
C_values = [0.1, 1.0]
log_rows = []
log_fitted = {}

for C in C_values:
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(C=C, max_iter=2000, solver="lbfgs"))
    ])
    clf.fit(X_train, y_train_bin)

    val_proba = clf.predict_proba(X_val)[:, 1]
    val_pred  = (val_proba >= 0.5).astype(int)

    row = {
        "model": f"LogReg(C={C})",
        "val_ROC_AUC": roc_auc_score(y_val_bin, val_proba),
        "val_LogLoss": log_loss(y_val_bin, val_proba),
        "val_Accuracy": accuracy_score(y_val_bin, val_pred)
    }
    log_rows.append(row)
    log_fitted[C] = clf

log_tbl = pd.DataFrame(log_rows).sort_values(
    ["val_ROC_AUC", "val_LogLoss"],
    ascending=[False, True]
).reset_index(drop=True)

display(log_tbl)

# 3) Select best model by validation ROC-AUC (tie-break: lowest log-loss)
best_name = log_tbl.iloc[0]["model"]
best_C = float(best_name.split("C=")[1].replace(")", ""))
best_clf = log_fitted[best_C]

print(f"\n✅ Best Logistic model (by validation): {best_name}")

# 4) Evaluate best Logistic model on TEST
test_proba = best_clf.predict_proba(X_test)[:, 1]
test_pred  = (test_proba >= 0.5).astype(int)

test_roc  = roc_auc_score(y_test_bin, test_proba)
test_ll   = log_loss(y_test_bin, test_proba)
test_acc  = accuracy_score(y_test_bin, test_pred)

print("\n📌 Test Results — Logistic Regression Screening")
print(f"Test ROC-AUC  : {test_roc:.4f}")
print(f"Test Log-Loss : {test_ll:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")

# 5) Confusion Matrix Plot + Screening Interpretation Note
cm = confusion_matrix(y_test_bin, test_pred)
cm_df = pd.DataFrame(cm, index=["Actual Low", "Actual High"], columns=["Pred Low", "Pred High"])

fig = px.imshow(
    cm_df,
    text_auto=True,
    title="Confusion Matrix — Logistic Regression Screening (High vs Low Risk)",
    labels=dict(x="Predicted Class", y="Actual Class")
)

fig.add_annotation(
    text=("Healthcare screening note: False Negatives (Actual High → Pred Low) are riskier "
          "because high-risk patients may be missed. False Positives increase follow-up workload."),
    xref="paper", yref="paper", x=0, y=1.10,
    showarrow=False,
    font=dict(size=13)
)

fig.update_layout(width=900, height=520, margin=dict(l=80, r=40, t=90, b=60))
fig.show()

### Additional Linear Models (Regularized Regression)

Although Logistic Regression is typically used for classification problems, this lab focuses on predicting a continuous outcome (disease progression). Therefore, instead of logistic regression, we implement two regularized linear regression models:

1. Ridge Regression
2. Lasso Regression

These models help control overfitting by adding regularization penalties while still allowing evaluation using R², MAE, and MAPE — consistent with the rubric requirements.

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

# Define models
ridge = Ridge(alpha=1.0)
lasso = Lasso(alpha=0.1)

models = {
    "Ridge Regression": ridge,
    "Lasso Regression": lasso
}

results_part3 = []

for name, model in models.items():
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)
    
    results_part3.append({
        "Model": name,
        "Train R2": r2_score(y_train, y_train_pred),
        "Val R2": r2_score(y_val, y_val_pred),
        "Test R2": r2_score(y_test, y_test_pred),
        "Train MAE": mean_absolute_error(y_train, y_train_pred),
        "Val MAE": mean_absolute_error(y_val, y_val_pred),
        "Test MAE": mean_absolute_error(y_test, y_test_pred),
        "Train MAPE": mean_absolute_percentage_error(y_train, y_train_pred),
        "Val MAPE": mean_absolute_percentage_error(y_val, y_val_pred),
        "Test MAPE": mean_absolute_percentage_error(y_test, y_test_pred)
    })

import pandas as pd
results_part3_df = pd.DataFrame(results_part3)
results_part3_df

The decision tree with max_depth=6 exhibits clear overfitting, 
with a large gap between training R² (0.79) and validation R² (0.16). 
This reflects high variance behavior.

Similarly, kNN with small k (k=3) shows strong training performance but weaker validation performance, 
indicating sensitivity to noise.

In contrast, kNN (k=15) demonstrates better generalization stability.

# Part 3 Conclusion 

## Why we are doing this
Because diabetes can worsen over time, we want an early-warning system that helps doctors **spot patients who may progress faster**, so they can intervene sooner.

## What we did
We tested multiple model types using the same fair split:
- Regression models (polynomial, trees, kNN) to predict an actual progression score
- Logistic regression as a screening classifier to flag “high-risk vs low-risk” patients

## What we achieved
- Found the best multivariate regression model based on validation results and confirmed it on the test set.
- Built a screening classifier (logistic regression) and measured how well it detects high-risk patients using ROC-AUC and log-loss.

## Key limitations
- These models learn patterns from a curated dataset and may not generalize perfectly to new hospitals or populations.
- In real-world use, we would need external validation, fairness checks, calibration, and ongoing monitoring.

Among the regularized linear models, Lasso Regression achieved the strongest validation performance with the highest R² and lowest MAE and MAPE. This indicates that feature selection through L1 regularization improved generalization on unseen validation data.

However, both Ridge and Lasso exhibit a noticeable drop in performance from validation to test sets, suggesting some degree of overfitting or variance instability due to dataset size. This highlights the importance of cross-validation and careful hyperparameter tuning in clinical screening applications.

Overall, Lasso provides the best trade-off between bias and variance in this experiment.

### Final Model Selection (Multivariate)

We compare all multivariate regression models using validation performance 
(highest validation R², tie-break: lowest validation MAE).

The selected model is: kNN (k=15).

This model provides the strongest validation R² while maintaining moderate error values, 
indicating a better bias–variance trade-off compared to deeper trees and lower k values.

## Exploratory Data Analysis (EDA)

EDA helps us understand the dataset before modeling.  
We use EDA to check data quality (missing values/outliers), understand distributions, and find relationships between features and the target.

### Dataset Overview

This dataset contains baseline patient measurements (10 numeric features) and a target:
**“disease progression one year after baseline.”**

EDA goals:
- Understand feature distributions (histograms)
- See feature-target relationships (scatter plots)
- Identify correlated variables (correlation matrix)
- Check if cleaning is required

### Key EDA Insights (What we learned)

1) Some features show stronger association with progression (example: BMI often shows a visible relationship).  
2) Several features can be correlated with each other, so multivariate models may outperform single-feature models.  
3) Distributions may be skewed for some features, which can influence model error and interpretation.  
4) If missing values and duplicates are absent (as expected in a curated dataset), aggressive cleaning is not needed.  
5) Correlation is not causation — EDA shows relationships, but the model is still predictive, not explanatory.

### EDA Visuals & Statistics

The next cell:
- Prints summary statistics and missing values
- Shows target distribution and feature distributions
- Shows scatter plots for key features vs target
- Shows correlation matrix for overall relationships

In [ ]:
"""
This cell builds a single premium EDA dashboard in a 2×2 layout:
(1) Target distribution
(2) Correlation heatmap
(3) BMI vs Target (with trendline)
(4) Risk bands: box plot of BMI by risk level
"""

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Theme (dark / elegant) ---
px.defaults.template = "plotly_dark"

# Risk bands for “screening-style” visuals
df_eda = df.copy()
df_eda["risk_band"] = pd.qcut(df_eda["target"], q=3, labels=["Low", "Medium", "High"])

# Correlation
corr = df_eda.drop(columns=["risk_band"]).corr(numeric_only=True)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Target Distribution (Progression in 1 Year)",
        "Correlation Heatmap (Features + Target)",
        "BMI vs Target (Trendline shows direction)",
        "BMI by Risk Band (Low / Medium / High)"
    )
)

# (1) Target histogram
hist = px.histogram(df_eda, x="target", nbins=35)
for tr in hist.data:
    fig.add_trace(tr, row=1, col=1)

# (2) Correlation heatmap
hm = go.Heatmap(
    z=corr.values,
    x=corr.columns,
    y=corr.index,
    zmin=-1, zmax=1,
    showscale=True
)
fig.add_trace(hm, row=1, col=2)

# (3) BMI vs Target scatter (colored by risk band)
sc = px.scatter(df_eda, x="bmi", y="target", color="risk_band", trendline="ols")
for tr in sc.data:
    fig.add_trace(tr, row=2, col=1)

# (4) BMI by risk band box plot
bx = px.box(df_eda, x="risk_band", y="bmi", points="all")
for tr in bx.data:
    fig.add_trace(tr, row=2, col=2)

# Layout polish (big title + consistent spacing)
fig.update_layout(
    title="Premium EDA Dashboard — Diabetes Progression Dataset",
    height=850,
    margin=dict(t=90, l=40, r=40, b=40),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    font=dict(size=14)
)

# Extra annotation like your sample
fig.add_annotation(
    text="How to read: Correlation hints predictive potential; validation will confirm what generalizes.",
    xref="paper", yref="paper", x=0, y=1.12, showarrow=False,
    font=dict(size=16)
)

fig.show()

In [ ]:
"""
This cell:
1) Trains a DecisionTreeRegressor on ALL features (Part 3 model).
2) Visualizes the TREE STRUCTURE (like the sample image).
3) Shows feature importance as a premium companion plot.
"""

from sklearn.tree import DecisionTreeRegressor, plot_tree
import matplotlib.pyplot as plt

# Train a tree for visualization (keep shallow for readability)
tree_vis = DecisionTreeRegressor(max_depth=3, random_state=42)
tree_vis.fit(X_train, y_train)

# ---- TREE STRUCTURE PLOT ----
plt.figure(figsize=(22, 10))
plot_tree(
    tree_vis,
    feature_names=X.columns,
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Decision Tree Regressor (max_depth=3) — Visual Structure for Screening Support", fontsize=18)
plt.show()

# ---- FEATURE IMPORTANCE (premium companion) ----
importances = pd.Series(tree_vis.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10,6))
importances.head(10).plot(kind="bar")
plt.title("Top Feature Importances (Decision Tree)", fontsize=16)
plt.ylabel("Importance")
plt.show()

In [ ]:
# Save the decision tree figure as a high-quality PNG
plt.figure(figsize=(22, 10))
plot_tree(tree_vis, feature_names=X.columns, filled=True, rounded=True, fontsize=10)
plt.title("Decision Tree Regressor (max_depth=3) — Visual Structure", fontsize=18)
plt.tight_layout()
plt.savefig("decision_tree_regressor_depth3.png", dpi=300, bbox_inches="tight")
plt.show()

print("✅ Saved: decision_tree_regressor_depth3.png")

In [ ]:
"""
This cell:
1) Trains a DecisionTreeRegressor for Part 3 using all features.
2) Keeps max_depth small so the tree is interpretable and printable.
"""

from sklearn.tree import DecisionTreeRegressor

tree_model = DecisionTreeRegressor(max_depth=4, min_samples_leaf=10, random_state=42)
tree_model.fit(X_train, y_train)

print("✅ Trained DecisionTreeRegressor(max_depth=4, min_samples_leaf=10)")

In [ ]:
"""
TREE SPLIT STORYTELLING (Premium Interpretability Panel)

This cell:
1) Extracts sample membership for each tree node using decision_path.
2) For the top split nodes (by depth + sample size), it plots:
   - Parent node target distribution
   - Left child target distribution
   - Right child target distribution
3) Adds split rules and summary stats directly on the plot.
"""

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -------------------------
# 1) Helpers
# -------------------------
tree = tree_model.tree_
feature_names = list(X.columns)

# Sparse decision path for each sample to each node
node_path = tree_model.decision_path(X_train)  # shape: (n_samples, n_nodes)

def node_samples(node_id):
    """Return indices of samples that reach node_id."""
    # column slice gives which samples go through that node
    col = node_path[:, node_id].toarray().ravel().astype(bool)
    return np.where(col)[0]

def node_rule(node_id):
    """Human-readable split rule for internal nodes."""
    f = tree.feature[node_id]
    t = tree.threshold[node_id]
    if f == -2:  # leaf
        return "Leaf"
    return f"{feature_names[f]} ≤ {t:.3f}"

def node_depths():
    """Compute depth for each node (BFS)."""
    depths = np.zeros(tree.node_count, dtype=int)
    stack = [(0, 0)]
    while stack:
        node, depth = stack.pop()
        depths[node] = depth
        left = tree.children_left[node]
        right = tree.children_right[node]
        if left != -1:
            stack.append((left, depth + 1))
        if right != -1:
            stack.append((right, depth + 1))
    return depths

depths = node_depths()

# Internal nodes only
internal_nodes = [i for i in range(tree.node_count) if tree.children_left[i] != -1]

# Rank nodes: prefer shallow (more important) + lots of samples
node_stats = []
for nid in internal_nodes:
    idx = node_samples(nid)
    node_stats.append({
        "node": nid,
        "depth": depths[nid],
        "n_samples": len(idx),
        "rule": node_rule(nid)
    })

node_stats = pd.DataFrame(node_stats)

# Choose which nodes to visualize (top splits)
# You can adjust: show_nodes = node_stats.sort_values([...]).head(K)
K = 6
show_nodes = (
    node_stats.sort_values(["depth", "n_samples"], ascending=[True, False])
             .head(K)["node"].tolist()
)

print("✅ Visualizing split nodes:", show_nodes)

# -------------------------
# 2) Build storytelling panel: rows = nodes, cols = Parent/Left/Right
# -------------------------
fig = make_subplots(
    rows=K, cols=3,
    column_titles=["Parent Node (before split)", "Left Child (≤ threshold)", "Right Child (> threshold)"],
    vertical_spacing=0.06
)

for r, nid in enumerate(show_nodes, start=1):
    left = tree.children_left[nid]
    right = tree.children_right[nid]

    idx_parent = node_samples(nid)
    idx_left = node_samples(left)
    idx_right = node_samples(right)

    y_parent = y_train.iloc[idx_parent].values
    y_left = y_train.iloc[idx_left].values
    y_right = y_train.iloc[idx_right].values

    # Summary numbers
    parent_mean = np.mean(y_parent) if len(y_parent) else np.nan
    left_mean = np.mean(y_left) if len(y_left) else np.nan
    right_mean = np.mean(y_right) if len(y_right) else np.nan

    parent_med = np.median(y_parent) if len(y_parent) else np.nan
    left_med = np.median(y_left) if len(y_left) else np.nan
    right_med = np.median(y_right) if len(y_right) else np.nan

    # Use violin plots (premium + distribution shape)
    fig.add_trace(go.Violin(
        y=y_parent, box_visible=True, meanline_visible=True,
        name=f"node {nid} parent", showlegend=False
    ), row=r, col=1)

    fig.add_trace(go.Violin(
        y=y_left, box_visible=True, meanline_visible=True,
        name=f"node {nid} left", showlegend=False
    ), row=r, col=2)

    fig.add_trace(go.Violin(
        y=y_right, box_visible=True, meanline_visible=True,
        name=f"node {nid} right", showlegend=False
    ), row=r, col=3)

    # Add split rule annotation at row start
    rule_txt = node_rule(nid)
    fig.add_annotation(
        text=f"<b>Split Node {nid}</b> (depth {depths[nid]}): {rule_txt}<br>"
             f"Parent n={len(y_parent)}, mean={parent_mean:.1f}, median={parent_med:.1f}",
        xref="paper", yref="paper",
        x=0.01, y=1 - ((r-1) / K) - 0.03,
        showarrow=False,
        font=dict(size=13)
    )

    # Add child stats annotations
    fig.add_annotation(
        text=f"Left n={len(y_left)}<br>mean={left_mean:.1f}, med={left_med:.1f}",
        xref="paper", yref="paper",
        x=0.52, y=1 - ((r-1) / K) - 0.03,
        showarrow=False,
        font=dict(size=12)
    )
    fig.add_annotation(
        text=f"Right n={len(y_right)}<br>mean={right_mean:.1f}, med={right_med:.1f}",
        xref="paper", yref="paper",
        x=0.86, y=1 - ((r-1) / K) - 0.03,
        showarrow=False,
        font=dict(size=12)
    )

# Layout polish (dark dashboard style)
fig.update_layout(
    template="plotly_dark",
    height=300*K,
    title="Decision Tree Split Storytelling — How Each Split Changes Target Distribution (Train Set)",
    margin=dict(t=90, l=40, r=40, b=40),
    font=dict(size=13)
)

fig.add_annotation(
    text="Interpretation: a good split separates target distributions (left vs right) — helpful for screening logic.",
    xref="paper", yref="paper", x=0, y=1.06, showarrow=False, font=dict(size=15)
)

fig.show()

# Final Conclusion

## Why we are doing this
We want to **predict diabetes progression risk early**, so physicians can **spot high-risk patients sooner** and act before the condition worsens.

## What we are doing
We tested multiple model types (polynomial regression, decision trees, kNN, and logistic regression screening) and compared them using proper metrics and a proper train/validation/test split.

## What we achieved at the end
- We built models that can estimate progression **one year after baseline**.
- We selected models using validation results (to avoid cheating) and tested only once at the end.
- We learned where models fail (especially for extreme cases), and we documented limitations so the model is used as a **support tool**, not as a diagnosis.

Regularization improved generalization performance by penalizing large coefficients.
Ridge shrinks coefficients smoothly, while Lasso can drive some coefficients to zero, effectively performing feature selection.
This highlights the trade-off between bias and variance in multivariate regression models.

The decision tree with max_depth=6 exhibits clear overfitting, with a large gap between training R² (0.79) and validation R² (0.16). This demonstrates high variance behavior. In contrast, kNN (k=15) provides a better bias–variance balance and generalizes more consistently to unseen validation data.

## Limitations 
This dataset is small and curated. Real hospital deployment would require external validation, fairness checks, calibration, and ongoing monitoring for data drift.